# 03. 학습 - SOLAR-10.7B-Instruct (QLoRA)

`upstage/SOLAR-10.7B-Instruct` 기반 QLoRA 파인튜닝  
WandB / Notion Training Datasheet 자동 기록

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
from dotenv import load_dotenv
load_dotenv('../.env')
print('환경 설정 완료')

---
## 1. 하이퍼파라미터 설정

### 주요 파라미터 가이드

| 파라미터 | 기본값 | 설명 |
|---|---|---|
| `lora.r` | 16 | LoRA rank (8/16/32/64), 높을수록 파라미터↑ |
| `lora.alpha` | 16 | 보통 r과 동일하게 설정 |
| `learning_rate` | 2e-4 | QLoRA 권장 범위: 1e-4 ~ 3e-4 |
| `num_epochs` | 3 | 과적합 주의, 3~5 권장 |
| `batch_size × grad_accum` | 1 × 16 = 16 | 실효 배치 크기 |
| `response_only` | True | 요약 부분만 loss 계산 (권장) |

In [ ]:
from src.training.trainer import (
    ModelConfig, LoraConfig, TrainConfig,
    ExperimentConfig, HyperParams,
)

# ============================================================
#  하이퍼파라미터 설정  <- 여기를 수정하세요
# ============================================================
model_cfg = ModelConfig(
    model_name       = "upstage/SOLAR-10.7B-Instruct",
    max_seq_length   = 2048,
    load_in_4bit     = True,
)

lora_cfg = LoraConfig(
    r                = 16,
    alpha            = 16,
    dropout          = 0.0,
    target_modules   = ["q_proj", "k_proj", "v_proj", "o_proj",
                        "gate_proj", "up_proj", "down_proj"],
)

train_cfg = TrainConfig(
    num_epochs       = 3,
    batch_size       = 1,
    grad_accum       = 16,
    learning_rate    = 2e-4,
    lr_scheduler     = "cosine",
    warmup_ratio     = 0.05,
    weight_decay     = 0.01,
    bf16             = True,
    seed             = 42,
    response_only    = True,
    max_new_tokens   = 256,
    instruction_part = "### User:\n",
    response_part    = "### Assistant:\n",
)

# ============================================================
#  실험 정보  <- 매 실험마다 수정하세요
# ============================================================
exp_cfg = ExperimentConfig(
    run_name   = "solar-10.7b-qlora-r16-ep3",
    dataset    = "DialogSum-KO",
    purpose    = "SOLAR-10.7B-Instruct QLoRA 베이스라인 실험",
    tags       = ["SOLAR", "causal-lm", "QLoRA"],
    output_dir = "../outputs/checkpoints",
)

hp = HyperParams(
    model=model_cfg, lora=lora_cfg,
    train=train_cfg, experiment=exp_cfg,
)
hp.summary()

---
## 2. 프롬프트 템플릿

SOLAR-10.7B-Instruct의 Chat 형식에 맞춘 프롬프트

In [ ]:
SOLAR_PROMPT = """### User:
다음 대화를 한국어로 간결하게 요약하세요.

{dialogue}

### Assistant:
{summary}"""

print("프롬프트 템플릿 예시:")
print(SOLAR_PROMPT.format(dialogue="A: 안녕하세요!\nB: 네, 안녕하세요!", summary="A와 B가 인사를 나눴다."))

---
## 3. WandB / Notion 연동

In [ ]:
import yaml
from src.utils.wandb_logger import WandbLogger
from src.utils.notion_logger import NotionLogger

with open('../configs/solar_config.yaml') as f:
    base_config = yaml.safe_load(f)

base_config['wandb']['name'] = exp_cfg.run_name
base_config['wandb']['tags'] = exp_cfg.tags

wandb_logger  = WandbLogger(base_config)
notion_logger = NotionLogger()

print(f'WandB run: {wandb_logger.run.url}')

---
## 4. 데이터 로드

In [ ]:
train_df = pd.read_csv('../data/raw/train.csv')
dev_df   = pd.read_csv('../data/raw/dev.csv')
test_df  = pd.read_csv('../data/raw/test.csv')

print(f'train: {len(train_df):,}  |  dev: {len(dev_df):,}  |  test: {len(test_df):,}')

---
## 5. 학습 실행

In [ ]:
from src.training.trainer import DialogueSummarizationTrainer

trainer = DialogueSummarizationTrainer(
    hp             = hp,
    prompt_template= SOLAR_PROMPT,
    notion_logger  = notion_logger,
    wandb_logger   = wandb_logger,
)

scores = trainer.run(train_df, dev_df)

print('\n=== 최종 ROUGE 점수 (MeCab) ===')
for k, v in scores.items():
    print(f'  {k}: {v}')

---
## 6. 제출 파일 생성

In [ ]:
from src.inference.submit import SubmissionGenerator

predictions = trainer.predict(test_df['dialogue'].tolist()) if hasattr(trainer, 'predict') else []

# predict 메서드 직접 호출
from unsloth import FastLanguageModel
import torch

FastLanguageModel.for_inference(trainer.model)
predictions = []
for dialogue in test_df['dialogue']:
    prompt = SOLAR_PROMPT.format(dialogue=dialogue, summary="")
    inputs = trainer.tokenizer(prompt, return_tensors="pt").to("cuda")
    with torch.no_grad():
        outputs = trainer.model.generate(
            **inputs,
            max_new_tokens=hp.train.max_new_tokens,
            do_sample=False,
        )
    pred = trainer.tokenizer.decode(
        outputs[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True,
    ).strip()
    predictions.append(pred)

gen = SubmissionGenerator(
    sample_submission_path='../data/raw/sample_submission.csv',
    output_dir='../outputs/predictions',
)
submit_path = gen.save(predictions=predictions, filename=f"{exp_cfg.run_name}.csv")
print(f'제출 파일: {submit_path}')

---
## 7. 마무리

In [ ]:
wandb_logger.finish()
print('완료!')
print(f'WandB : {wandb_logger.run.url}')